# Terminal primer removal with Myers

This prototype removes primers from the adapter- and barcode-cleaned reads in `manual_test/processed/EQA_04.subsampled.fastq`. It uses Biopython for FASTA/FASTQ parsing, reverse complements, and IUPAC ambiguity values, while the matching step uses a pure-Python Myers bit-vector edit-distance scan.

It supports both read orientations. At the 5' end it searches supplied `*_LEFT` primers for forward reads and supplied `*_RIGHT` primers for reverse reads. At the 3' end it searches reverse complements of `*_RIGHT` primers for forward reads and reverse complements of `*_LEFT` primers for reverse reads. A primer must align within the configured terminal slack; internal matches are ignored.

The workflow searches 160-base terminal windows with up to 50 bases of terminal slack, requires at least `max(12, ceil(0.55 * primer_length))` aligned bases, and permits a 20% observed edit rate. Exact 7-base seeds select candidate placements before Myers verification. The notebook audits 250 reads, streams the 10,000-read subsample to `manual_test/processed/EQA_04.subsampled.noprimers.fastq`, and validates record order, sequence/quality synchronization, and non-increasing read lengths.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator
import gzip
import math

from Bio import SeqIO
from Bio.Data.IUPACData import ambiguous_dna_values
from Bio.Seq import Seq
from Bio.SeqIO.QualityIO import FastqGeneralIterator

In [ ]:
DIRECTORY_ROOT = Path.cwd()
INPUT_FASTQ = DIRECTORY_ROOT / "manual_test/processed/EQA_04.noadapters.fastq"
PRIMER_FASTA = DIRECTORY_ROOT / "manual_test/ESIB_EQA_2026_SARS1.primers.fasta"
OUTPUT_FASTQ = DIRECTORY_ROOT / "manual_test/processed/EQA_04.noprimers.bitvector.fastq"


### Search and trimming settings

In [ ]:
TERMINAL_WINDOW = 200 # NOTE: terminal_window and terminal_slack are currently fixed sizes but in a real implementation they should (probably) be determined on a per-read basis.
TERMINAL_SLACK = 170 # NOTE: first guesstimation would be to set terminal window to roughly 50% of the read-length and terminal slack to `terminal_window - max_primer_length`
MAX_ERROR_RATE = 0.30
MIN_ALIGNED_PRIMER_FRACTION = 0.27
MIN_ALIGNED_PRIMER_BASES = 8
OVERHANG_PENALTY = 0.5
SEED_LENGTH = 7

## Primer and FASTQ helpers

In [ ]:
@dataclass(frozen=True)
class Primer:
    name: str
    sequence: str


@dataclass(frozen=True)
class PreparedPrimer:
    primer: Primer
    orientation: str
    sequence: str
    eq_table: dict[str, int]


@dataclass(frozen=True)
class TerminalMatch:
    primer: Primer
    orientation: str
    start: int
    end: int
    cost: int
    aligned_bases: int


def read_fastq(path: Path) -> Iterator[tuple[str, str, str]]:
    # Read FASTQ records lazily and keep sequence and quality strings synchronized.
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt") as handle:
        for title, sequence, qualities in FastqGeneralIterator(handle):
            yield title, sequence.upper(), qualities


def read_primers(path: Path) -> tuple[tuple[Primer, ...], tuple[Primer, ...]]:
    # Separate primers by their declared amplicon orientation.
    left_primers = []
    right_primers = []
    for record in SeqIO.parse(path, "fasta"):
        primer = Primer(record.id, str(record.seq).upper())
        if "_LEFT" in primer.name:
            left_primers.append(primer)
        elif "_RIGHT" in primer.name:
            right_primers.append(primer)
        else:
            raise ValueError(f"Primer name has no LEFT/RIGHT orientation: {primer.name}")

    if not left_primers or not right_primers:
        raise ValueError("Expected at least one left and one right primer")
    return tuple(left_primers), tuple(right_primers)


def minimum_aligned_bases(primer: Primer) -> int:
    # Require both an absolute minimum and a primer-length-relative overlap.
    return max(
        MIN_ALIGNED_PRIMER_BASES,
        math.ceil(len(primer.sequence) * MIN_ALIGNED_PRIMER_FRACTION),
    )


def build_eq_table(pattern: str) -> dict[str, int]:
    # Expand IUPAC symbols into the Myers equality bit masks.
    eq = {base: 0 for base in "ACGTN"}
    for index, symbol in enumerate(pattern.upper()):
        for base in ambiguous_dna_values.get(symbol, ""):
            eq[base] |= 1 << index
    eq["N"] = (1 << len(pattern)) - 1
    return eq

## Oriented primer indexes

In [ ]:
def prepare_primers(
    primers: tuple[Primer, ...],
    orientation: str,
    reverse_for_prefix_search: bool,
) -> tuple[PreparedPrimer, ...]:
    # Prepare the sequence orientation used by a prefix-style Myers scan.
    prepared = []
    for primer in primers:
        sequence = (
            str(Seq(primer.sequence).reverse_complement())
            if orientation == "reverse_complement"
            else primer.sequence
        )
        if reverse_for_prefix_search:
            sequence = sequence[::-1]
        prepared.append(
            PreparedPrimer(
                primer,
                orientation,
                sequence,
                build_eq_table(sequence),
            )
        )
    return tuple(prepared)


def build_seed_index(
    prepared_primers: tuple[PreparedPrimer, ...],
) -> dict[str, tuple[tuple[int, int], ...]]:
    # Index exact seeds to avoid running Myers against every primer on every read.
    index = defaultdict(list)
    for primer_index, prepared in enumerate(prepared_primers):
        for offset in range(len(prepared.sequence) - SEED_LENGTH + 1):
            seed = prepared.sequence[offset:offset + SEED_LENGTH]
            index[seed].append((primer_index, offset))
    return {seed: tuple(locations) for seed, locations in index.items()}

## Myers bit-vector matching and trimming

In [ ]:
def myers_global_scores(
    text: str,
    pattern: str,
    eq_table: dict[str, int] | None = None,
) -> list[int]:
    """Return Levenshtein distances from pattern to every text prefix."""
    if not pattern:
        raise ValueError("A primer pattern cannot be empty")

    # Initialize the bit-vector state for a global pattern-to-text alignment.
    eq_table = eq_table if eq_table is not None else build_eq_table(pattern)
    high_bit = 1 << (len(pattern) - 1)
    positive = ~0
    negative = 0
    score = len(pattern)
    scores = []

    for base in text:
        # Apply one text base to the Myers vertical and horizontal states.
        matched = eq_table.get(base, 0)
        vertical = matched | negative
        horizontal = (((matched & positive) + positive) ^ positive) | matched
        positive_horizontal = negative | ~(horizontal | positive)
        negative_horizontal = positive & horizontal

        # Track the edit distance at the pattern's final bit.
        if positive_horizontal & high_bit:
            score += 1
        elif negative_horizontal & high_bit:
            score -= 1

        positive_horizontal = (positive_horizontal << 1) | 1
        negative_horizontal <<= 1
        positive = negative_horizontal | ~(vertical | positive_horizontal)
        negative = positive_horizontal & vertical
        scores.append(score)
    return scores


def seed_supported_candidates(
    text: str,
    prepared_primers: tuple[PreparedPrimer, ...],
    seed_index: dict[str, tuple[tuple[int, int], ...]],
) -> set[tuple[int, int, int]]:
    # Convert exact seed hits into possible primer shifts at this read boundary.
    candidates = set()
    maximum_pattern_length = max(len(prepared.sequence) for prepared in prepared_primers)
    scan_length = min(len(text), maximum_pattern_length + TERMINAL_SLACK)
    for text_offset in range(scan_length - SEED_LENGTH + 1):
        seed = text[text_offset:text_offset + SEED_LENGTH]
        for primer_index, primer_offset in seed_index.get(seed, ()):
            prepared = prepared_primers[primer_index]
            shift = text_offset - primer_offset
            minimum_overlap = minimum_aligned_bases(prepared.primer)
            if -(len(prepared.sequence) - minimum_overlap) <= shift <= TERMINAL_SLACK:
                if shift < 0:
                    candidates.add((primer_index, -shift, 0))
                else:
                    candidates.add((primer_index, 0, shift))
    return candidates


def candidate_matches(
    text: str,
    prepared: PreparedPrimer,
    suffix_start: int,
    text_start: int,
) -> list[TerminalMatch]:
    # Verify one seed-supported overlap with Myers edit distance.
    suffix = prepared.sequence[suffix_start:]
    overhang_cost = math.ceil(suffix_start * OVERHANG_PENALTY)
    eq_table = prepared.eq_table if suffix_start == 0 else build_eq_table(suffix)
    maximum_errors = math.ceil(len(suffix) * MAX_ERROR_RATE)
    maximum_length = min(len(text) - text_start, len(suffix) + maximum_errors)
    candidates = []

    for relative_end, edit_distance in enumerate(
        myers_global_scores(
            text[text_start:text_start + maximum_length],
            suffix,
            eq_table,
        )
    ):
        aligned_bases = relative_end + 1
        observed_error_budget = math.ceil(aligned_bases * MAX_ERROR_RATE)
        if (
            aligned_bases >= minimum_aligned_bases(prepared.primer)
            and edit_distance <= observed_error_budget
        ):
            candidates.append(
                TerminalMatch(
                    prepared.primer,
                    prepared.orientation,
                    text_start,
                    text_start + relative_end,
                    edit_distance + overhang_cost,
                    aligned_bases,
                )
            )
    return candidates


def match_rank(match: TerminalMatch) -> tuple[float, int, int]:
    # Prefer larger primer coverage, then lower cost and a nearer boundary.
    return (
        -(match.aligned_bases / len(match.primer.sequence)),
        match.cost,
        match.start,
    )


def best_terminal_match(
    text: str,
    prepared_primers: tuple[PreparedPrimer, ...],
    seed_index: dict[str, tuple[tuple[int, int], ...]],
) -> TerminalMatch | None:
    # Search only candidates supported by an exact seed, then verify with Myers.
    candidates = [
        candidate
        for primer_index, suffix_start, text_start in seed_supported_candidates(
            text,
            prepared_primers,
            seed_index,
        )
        for candidate in candidate_matches(
            text,
            prepared_primers[primer_index],
            suffix_start,
            text_start,
        )
    ]
    if not candidates:
        return None

    # Reject equally placed matches belonging to different primer identities.
    best = min(candidates, key=lambda candidate: (candidate.cost, candidate.start))
    tied_primers = {
        candidate.primer.name
        for candidate in candidates
        if (candidate.cost, candidate.start) == (best.cost, best.start)
    }
    if len(tied_primers) > 1:
        return None
    return best


def trim_primers(sequence: str, qualities: str):
    # Search each terminal in its natural prefix orientation.
    if not sequence:
        return sequence, qualities, None, None

    left_match = best_terminal_match(
        sequence[:TERMINAL_WINDOW],
        LEFT_PREPARED_PRIMERS,
        LEFT_SEED_INDEX,
    )
    right_match = best_terminal_match(
        sequence[-TERMINAL_WINDOW:][::-1],
        RIGHT_PREPARED_PRIMERS,
        RIGHT_SEED_INDEX,
    )
    left_cut = left_match.end + 1 if left_match else 0
    right_cut = len(sequence) - (right_match.end + 1) if right_match else len(sequence)

    # Resolve overlapping terminal calls by retaining the stronger match.
    if left_cut > right_cut:
        if match_rank(left_match) <= match_rank(right_match):
            right_match = None
            right_cut = len(sequence)
        else:
            left_match = None
            left_cut = 0
    return sequence[left_cut:right_cut], qualities[left_cut:right_cut], left_match, right_match

## Coordinate checks

These synthetic reads verify the Myers recurrence and coordinate convention used for trimming before real reads are processed. The setup then loads both primer groups and prepares the forward and reverse-complement search indexes.

In [ ]:
# Validate the Myers recurrence before processing real reads.
primer = "AACAAACCAACCAACTTTCGATCTC"
assert myers_global_scores(primer, primer)[-1] == 0
assert myers_global_scores(primer[:-1] + "T", primer)[-1] == 1
assert myers_global_scores(primer[5:], primer[5:])[-1] == 0

# Prepare original left/right groups for both read orientations.
LEFT_PRIMERS, RIGHT_PRIMERS = read_primers(PRIMER_FASTA)
LEFT_PREPARED_PRIMERS = prepare_primers(
    LEFT_PRIMERS + RIGHT_PRIMERS,
    "forward",
    False,
)
RIGHT_PREPARED_PRIMERS = prepare_primers(
    RIGHT_PRIMERS + LEFT_PRIMERS,
    "reverse_complement",
    True,
)
LEFT_SEED_INDEX = build_seed_index(LEFT_PREPARED_PRIMERS)
RIGHT_SEED_INDEX = build_seed_index(RIGHT_PREPARED_PRIMERS)

# Verify that both terminal coordinates remove only the primer sequence.
left_primer = LEFT_PRIMERS[0]
right_primer = RIGHT_PRIMERS[0]
payload = "GATTACA" * 20
synthetic_sequence = (
    left_primer.sequence
    + payload
    + str(Seq(right_primer.sequence).reverse_complement())
)
synthetic_trimmed, _, synthetic_left, synthetic_right = trim_primers(
    synthetic_sequence,
    "I" * len(synthetic_sequence),
)
assert synthetic_left is not None and synthetic_left.primer == left_primer
assert synthetic_right is not None and synthetic_right.primer == right_primer
assert synthetic_trimmed == payload
print(
    f"Loaded {len(LEFT_PRIMERS)} left and {len(RIGHT_PRIMERS)} right primers; "
    f"input has {sum(1 for _ in read_fastq(INPUT_FASTQ)):,} reads."
)

## Inspect a small sample

This evaluates the first 250 reads from the input without writing output files. The summary shows how often each terminal is called and prints a few representative coordinate decisions.

In [ ]:
from collections import Counter
from itertools import islice

sample_decisions = [
    (title, sequence, trimmed, left_match, right_match)
    for title, sequence, qualities in islice(read_fastq(INPUT_FASTQ), 250)
    for trimmed, _, left_match, right_match in [trim_primers(sequence, qualities)]
]
left_calls = Counter(
    left_match.primer.name for *_, left_match, _ in sample_decisions if left_match
)
right_calls = Counter(
    right_match.primer.name for *_, right_match in sample_decisions if right_match
)
print(
    f"Audited reads: {len(sample_decisions)}\n"
    f"Left terminal calls: {sum(left_calls.values())}\n"
    f"Right terminal calls: {sum(right_calls.values())}\n"
    f"Both terminals called: {sum(bool(left and right) for *_, left, right in sample_decisions)}\n"
    f"Most common left primers: {left_calls.most_common(5)}\n"
    f"Most common right primers: {right_calls.most_common(5)}"
)

shown = 0
for title, sequence, trimmed, left_match, right_match in sample_decisions:
    if left_match or right_match:
        print(
            title.split()[0],
            f"{len(sequence)} -> {len(trimmed)}",
            f"left={left_match}",
            f"right={right_match}",
            sep=" | ",
        )
        shown += 1
        if shown == 5:
            break

## Write primer-cleaned FASTQ

Running this cell processes the full FASTQ file. Source files are preserved, output records remain in input order, and sequence and quality strings are trimmed together.

In [ ]:
total_reads = 0
left_trimmed = 0
right_trimmed = 0
both_trimmed = 0
bases_removed = 0
left_primers = Counter()
right_primers = Counter()

with OUTPUT_FASTQ.open("w") as output_handle:
    for title, sequence, qualities in read_fastq(INPUT_FASTQ):
        trimmed_sequence, trimmed_qualities, left_match, right_match = trim_primers(
            sequence, qualities,
        )
        assert len(trimmed_sequence) == len(trimmed_qualities)
        assert len(trimmed_sequence) <= len(sequence)
        output_handle.write(
            f"@{title}\n{trimmed_sequence}\n+\n{trimmed_qualities}\n"
        )

        total_reads += 1
        left_trimmed += left_match is not None
        right_trimmed += right_match is not None
        both_trimmed += left_match is not None and right_match is not None
        bases_removed += len(sequence) - len(trimmed_sequence)
        if left_match:
            left_primers[left_match.primer.name] += 1
        if right_match:
            right_primers[right_match.primer.name] += 1

print(
    f"Input reads: {total_reads:,}\n"
    f"Left terminal primers removed: {left_trimmed:,}\n"
    f"Right terminal primers removed: {right_trimmed:,}\n"
    f"Both terminal primers removed: {both_trimmed:,}\n"
    f"Bases removed: {bases_removed:,}\n"
    f"Most common left primers: {left_primers.most_common(10)}\n"
    f"Most common right primers: {right_primers.most_common(10)}\n"
    f"Output: {OUTPUT_FASTQ.relative_to(DIRECTORY_ROOT)}"
)

## Output validation

Validate the streamed FASTQ after writing: all source records must remain in order, sequence and quality lengths must remain synchronized, and trimming may only shorten reads.

In [ ]:
from itertools import zip_longest

validated_reads = 0
validated_removed_bases = 0
for source, cleaned in zip_longest(
    read_fastq(INPUT_FASTQ), read_fastq(OUTPUT_FASTQ), fillvalue=None,
):
    assert source is not None and cleaned is not None
    source_title, source_sequence, _ = source
    cleaned_title, cleaned_sequence, cleaned_qualities = cleaned
    assert cleaned_title == source_title
    assert len(cleaned_sequence) == len(cleaned_qualities)
    assert len(cleaned_sequence) <= len(source_sequence)
    validated_reads += 1
    validated_removed_bases += len(source_sequence) - len(cleaned_sequence)

assert validated_reads == total_reads
assert validated_removed_bases == bases_removed
assert OUTPUT_FASTQ.stat().st_size > 0
print(
    f"Validated {validated_reads:,} records and "
    f"{validated_removed_bases:,} removed bases in {OUTPUT_FASTQ.name}."
)